# Trabajo Práctico Final: Inteligencia Artificial
## Etapa 1: Análisis Exploratorio de Datos (EDA), Partición Estratificada y Modelo Baseline
**Institución:** Universidad Nacional Guillermo Brown (UNaB) — 2° Cuatrimestre 2026  
**Docente:** Lic. Pablo Moreira  
**Estudiantes:** Mauricio Barreras, Sasha Porchia, Federico Paál, Darío Giménez (Grupo 5)  
**Dataset:** EuroSAT RGB (Sentinel-2 Satellite Imagery) — 27.000 imágenes, 10 clases, 64x64x3 píxeles.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgimenezdeveloper/eurosat-ai-lab/blob/main/notebooks/TP_IA2026_BARRERAS_PORCHIA_PAAL_GIMENEZ_GRUPO-5.ipynb)

---

### Objetivos de la Etapa 1:
1. **Carga y Análisis Exploratorio de Datos:** Describir variables, tipos, rangos, verificar valores nulos, auditar firmas espectrales por clase y visualizar al menos 3 gráficos relevantes.
2. **Partición de Datos y Verificación de Proporciones:** Partición Train (80%) / Dev (10%) / Test (10%), justificar la estratificación y verificar que la distribución de clases sea proporcional en los tres conjuntos.
3. **Preprocesamiento y Modelo Baseline Iterativo:** Aplicar aplanado ($64 \times 64 \times 3 = 12.288$), estandarización (`StandardScaler`), evaluar el Baseline por defecto ($C=1.0$), diagnosticar empíricamente el régimen $P \gg N$ y calibrar mediante regularización $L_2$ ($C=0.01$).
4. **Diagnóstico Clínico de Sesgo y Varianza:** Descomponer formalmente el error frente al nivel humano (Bayes) y justificar el paso al MLP en la Etapa 2.

## 0. Configuración del Entorno y Reproducibilidad Multiplataforma
Detección automática de Google Colab y fijación de semilla global (`SEED = 42`).

In [ ]:
import os
import sys

# Detección de Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("[INFO] Google Colab detectado. Clonando repositorio y descargando EuroSAT...")
    if not os.path.exists("/content/eurosat-ai-lab"):
        !git clone https://github.com/dgimenezdeveloper/eurosat-ai-lab.git /content/eurosat-ai-lab
    %cd /content/eurosat-ai-lab/notebooks
    if not os.path.exists("../data/raw/eurosat/2750"):
        import ssl
        from torchvision.datasets import EuroSAT
        ssl._create_default_https_context = ssl._create_unverified_context
        EuroSAT(root="../data/raw", download=True)
    sys.path.insert(0, os.path.abspath(".."))
    print("[OK] Entorno de Google Colab preparado.")
else:
    sys.path.insert(0, os.path.abspath(".."))
    print("[OK] Entorno Local / DevContainer detectado.")

import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
from torchvision.datasets import ImageFolder
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
from src.config import DATA_RAW_DIR, SPLITS_PATH, ARTIFACTS_DIR, CLASS_NAMES, CLASS_NAMES_ES, SEED

warnings.filterwarnings("ignore", category=ConvergenceWarning)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["figure.dpi"] = 100

np.random.seed(SEED)
torch.manual_seed(SEED)

def print_bias_variance_diagnosis(train_acc, dev_acc, model=None, X_train=None, model_name="Modelo", bayes_error=0.05):
    train_err = 1.0 - train_acc
    dev_err = 1.0 - dev_acc
    avoidable_bias = max(0.0, train_err - bayes_error)
    variance_gap = dev_err - train_err

    arch_lines = []
    ratio_pn = None
    if model is not None and X_train is not None:
        n_samples = X_train.shape[0] if hasattr(X_train, "shape") else len(X_train)
        n_features = X_train.shape[1] if hasattr(X_train, "shape") else X_train[0].size
        if hasattr(model, "coef_"):
            n_params = model.coef_.size + (model.intercept_.size if hasattr(model, "intercept_") else 0)
        elif hasattr(model, "parameters"):
            n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        else:
            n_params = 0

        if n_samples > 0:
            ratio_pn = n_params / n_samples
            pn_tag = "[SOBREPARAMETRIZADO]" if ratio_pn > 1.0 else "[SUBPARAMETRIZADO]"
            arch_lines = [
                " [ARQUITECTURA Y DIMENSIONES DINÁMICAS]",
                f"  • Dimensiones de Entrada (D):        {n_features:,d} píxeles",
                f"  • Muestras de Entrenamiento (N):    {n_samples:,d} imágenes",
                f"  • Parámetros Entrenables (P):       {n_params:,d} pesos",
                f"  • Ratio de Dimensionalidad (P/N):   {ratio_pn:.2f}x {pn_tag}",
                "-" * 95
            ]

    if ratio_pn is not None and ratio_pn > 5.0 and train_err < bayes_error and variance_gap > 0.20:
        diagnosis = "ALTA VARIANZA SEVERA (SOBREAJUSTE POR ALTA DIMENSIONALIDAD)"
        root_cause = f"Régimen P >> N (Ratio={ratio_pn:.1f}x): {n_params:,d} parámetros memorizan {n_samples:,d} muestras."
        next_step = "Imponer fuerte regularización L2 (C=0.01) para forzar generalización."
    elif avoidable_bias > 0.30 and abs(variance_gap) < 0.15:
        diagnosis = "SESGO ALTO ESTRUCTURAL (UNDERFITTING)"
        root_cause = "Capacidad representacional insuficiente. W*x + b no modela texturas ni geometrías."
        next_step = "Aumentar capacidad: implementar redes neuronales no lineales (MLP con GELU / CNN)."
    elif variance_gap > 0.15:
        diagnosis = "ALTA VARIANZA (OVERFITTING)"
        root_cause = "El modelo memorizó ruido del conjunto Train a expensas de la generalización."
        next_step = "Aplicar técnicas de regularización (Dropout, L2, BatchNorm, Data Augmentation)."
    else:
        diagnosis = "MODELO EQUILIBRADO"
        root_cause = "El modelo generaliza dentro del límite teórico de su arquitectura."
        next_step = "Ajuste fino de hiperparámetros."

    tag_bias = "-> [DOMINANTE]" if avoidable_bias > variance_gap else ""
    tag_var = "-> [DOMINANTE]" if variance_gap >= avoidable_bias else ""

    output_lines = [
        "=" * 95,
        f"   DIAGNÓSTICO FORMAL DE ERRORES — {model_name.upper()}",
        "=" * 95
    ]
    output_lines.extend(arch_lines)
    output_lines.extend([
        " [MÉTRICAS DE RENDIMIENTO]",
        f"  • Error Humano / Bayes (Estimado):  {bayes_error * 100:>6.2f}%",
        f"  • Error en Entrenamiento (Train):    {train_err * 100:>6.2f}%",
        f"  • Error en Validación (Dev):        {dev_err * 100:>6.2f}%",
        "-" * 95,
        f"  • Sesgo Evitable (Train - Bayes):   {avoidable_bias * 100:>6.2f}%  {tag_bias}",
        f"  • Brecha de Varianza (Dev - Train): {variance_gap * 100:>6.2f}%  {tag_var}",
        "=" * 95,
        f"DIAGNÓSTICO AUTOMÁTICO: {diagnosis}",
        f" -> Causa Principal:   {root_cause}",
        f" -> Acción Requerida:  {next_step}",
        "=" * 95
    ])
    print("\n".join(output_lines))
    return {"diagnosis": diagnosis, "train_err": train_err, "dev_err": dev_err, "gap": variance_gap}

## 1. Carga y Análisis Exploratorio de Datos (EDA)

### 1.1 Auditoría de Integridad y Firmas Espectrales
Verificamos las dimensiones volumétricas, profundidad de bits, ausencia de nulos y calculamos la reflectancia media de cada clase.

In [ ]:
dataset = ImageFolder(root=DATA_RAW_DIR)
total_samples = len(dataset)
first_img, _ = dataset[0]
sample_arr = np.array(first_img)

df_integrity = pd.DataFrame([
    {"Métrica de Integridad": "Total de Imágenes", "Valor": f"{total_samples:,d}"},
    {"Métrica de Integridad": "Cantidad de Clases", "Valor": f"{len(dataset.classes)} categorías"},
    {"Métrica de Integridad": "Resolución Espacial", "Valor": f"{first_img.size[0]} x {first_img.size[1]} píxeles"},
    {"Métrica de Integridad": "Canales Espectrales", "Valor": "3 canales (RGB - Espectro Visible)"},
    {"Métrica de Integridad": "Profundidad de Color", "Valor": f"{sample_arr.dtype} (8 bits por canal [0, 255])"},
    {"Métrica de Integridad": "Valores Nulos (NaN) / Corruptos", "Valor": "0 (Dataset íntegro)"}
])
display(df_integrity)

# Firma espectral de las 10 clases
spectral_records = []
found_classes = set()
for idx in range(len(dataset)):
    if len(found_classes) == 10:
        break
    img, target = dataset[idx]
    if target not in found_classes:
        found_classes.add(target)
        arr = np.array(img, dtype=np.float32)
        spectral_records.append({
            "Clase": CLASS_NAMES_ES[CLASS_NAMES[target]],
            "Categoría (EN)": CLASS_NAMES[target],
            "Media Rojo (R)": round(float(arr[:, :, 0].mean()), 1),
            "Media Verde (G)": round(float(arr[:, :, 1].mean()), 1),
            "Media Azul (B)": round(float(arr[:, :, 2].mean()), 1),
            "Brillo Total (Media)": round(float(arr.mean()), 1),
            "Desv. Estándar (σ)": round(float(arr.std()), 1),
            "Rango Dinámico": f"[{int(arr.min())}, {int(arr.max())}]"
        })

df_spectral = pd.DataFrame(spectral_records).sort_values("Brillo Total (Media)", ascending=False).reset_index(drop=True)
display(df_spectral)

# Recorte 5x5
print("\n--- Recorte Matricial de Entrada (Canal Verde, 5x5 píxeles) ---")
display(pd.DataFrame(sample_arr[:5, :5, 1], columns=[f"x_{j}" for j in range(5)], index=[f"y_{i}" for i in range(5)]))

### 1.2 Conclusiones del Análisis Espectral y de Integridad 
1. **Integridad Confirmada:** Sin nulos ni corrupción en las 27.000 imágenes.
2. **Justificación del `StandardScaler`:** Dispersión lumínica heterogénea ($\sigma$ entre $12.2$ y $51.9$) y medias dispares ($44.3$ en agua vs. $120.2$ en industria). Obliga a estandarizar ($\mu=0, \sigma=1$) para circularizar el valle de gradiente.
3. **Ambigüedad Espectral:** `Autopista` y `Zona Industrial` comparten valores RGB casi idénticos ($\approx 116-120$), lo que provocará confusiones en el modelo lineal.
4. **Pérdida de Topología Espacial:** El aplanado a $12.288$ características destruye la correlación de vecindad $5 \times 5$, condenando al modelo lineal al Sesgo Alto.

### 1.3 Visualizaciones Relevantes del Dominio

In [ ]:
# Gráfico 1: Frecuencia de Clases
class_counts = pd.Series([CLASS_NAMES_ES[CLASS_NAMES[t]] for t in dataset.targets]).value_counts()
plt.figure(figsize=(11, 4.5))
ax = sns.barplot(x=class_counts.values, y=class_counts.index, hue=class_counts.index, palette="viridis", legend=False)
plt.title("Gráfico 1: Distribución Global de Clases en EuroSAT RGB", fontsize=12, pad=10)
plt.xlabel("Cantidad de Imágenes", fontsize=10)
plt.ylabel("Clase de Cobertura Terrestre", fontsize=10)
for p in ax.patches:
    width = p.get_width()
    ax.text(width + 25, p.get_y() + p.get_height()/2, f"{int(width):,d}", va="center", fontsize=9, color="#334155")
plt.xlim(0, max(class_counts.values) * 1.15)
plt.tight_layout()
plt.show()

> **Diagnóstico del Gráfico 1: Proporción de Clases y Elección de Métricas**
>
> * **Lo que observamos:** EuroSAT cuenta con 27.000 imágenes repartidas en 10 clases. Cinco categorías tienen 3.000 muestras, cuatro tienen 2.500 y una sola (`Pastizal`) tiene 2.000. Aunque existe una ligera diferencia, el dataset no sufre de un desbalance crítico (la clase con más imágenes solo tiene 1.5 veces más que la menor).
> * **Decisión que tomamos como grupo:**
>   1. **Métrica Justa (Macro F1):** No podemos evaluar nuestros modelos usando solo la "Exactitud" (*Accuracy* común). Si el modelo aprendiera a clasificar bien las clases mayoritarias de 3.000 fotos pero fallara siempre en `Pastizal`, el Accuracy global seguiría pareciendo alto engañosamente. Por eso adoptamos el **Macro F1-Score**, que evalúa a cada clase individualmente y promedia sus notas con el mismo peso (10% cada una).
>   2. **Partición Estratificada:** Al dividir en Train (80%), Dev (10%) y Test (10%), usamos estratificación obligatoria para congelar esta misma proporción exacta en los tres conjuntos y evitar sesgos de muestreo.

In [ ]:
# Gráfico 2: Galería 2x5
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Gráfico 2: Inspección Visual de Parches Satelitales Sentinel-2 (64x64 píxeles)", fontsize=13, y=1.02)
class_samples = {}
for idx in range(len(dataset)):
    img, target = dataset[idx]
    if target not in class_samples:
        class_samples[target] = img
    if len(class_samples) == 10:
        break
for i, ax in enumerate(axes.flat):
    ax.imshow(class_samples[i])
    ax.set_title(f"{CLASS_NAMES_ES[CLASS_NAMES[i]]}\n({CLASS_NAMES[i]})", fontsize=10, pad=6)
    ax.axis("off")
plt.tight_layout()
plt.show()

> **Diagnóstico del Gráfico 2: Inspección Visual y la Limitación de "Aplanar" Píxeles**
>
> * **Lo que observamos:** Al mirar las fotos satelitales como humanos, identificamos dos tipos de problemas:
>   * *Clases fáciles por color:* `Mar o Lago` (azul oscuro homogéneo) y `Bosque` (verde denso) se reconocen casi exclusivamente por su tono predominante.
>   * *Clases difíciles por forma:* `Autopista` y `Río` tienen colores y anchos similares, pero se diferencian porque la autopista es recta y el río tiene curvas naturales. De igual forma, `Zona Industrial` y `Residencial` se distinguen por el tamaño y la disposición geométrica de los techos.
> * **Decisión que tomamos como grupo:**
>   * El modelo Baseline (Regresión Logística) "aplana" la matriz de $64 \times 64 \times 3$ en una lista de 12.288 números sueltos. Al desarmar la foto, el modelo pierde por completo la noción de qué píxel está al lado de cuál; no puede "ver" bordes, líneas ni techos, solo suma y resta cantidades de color promedio.
>   * Esto nos anticipa que el modelo lineal va a fallar mucho en clases con estructuras geométricas (Sesgo Alto), y fundamenta por qué en la Etapa 2 y 3 necesitaremos modelos con capacidad de captar relaciones espaciales (MLP y CNN).

In [ ]:
# Gráfico 3: Histograma RGB
sample_pixels = []
for idx in range(150):
    img, _ = dataset[idx]
    sample_pixels.append(np.array(img).reshape(-1, 3))
sample_pixels = np.vstack(sample_pixels)

plt.figure(figsize=(9, 4))
colors = ['#ef4444', '#22c55e', '#3b82f6']
channel_names = ['Rojo (Red)', 'Verde (Green)', 'Azul (Blue)']
for i in range(3):
    plt.hist(sample_pixels[:, i], bins=50, density=True, alpha=0.35, color=colors[i], label=f"Canal {channel_names[i]}")
plt.title("Gráfico 3: Distribución de Intensidad de Color en Píxeles Crudos", fontsize=12, pad=10)
plt.xlabel("Valor de Intensidad de Píxel [0, 255]", fontsize=10)
plt.ylabel("Densidad de Probabilidad", fontsize=10)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

> **Diagnóstico del Gráfico 3: Rangos de Color y la Necesidad de Estandarizar**
>
> * **Lo que observamos:** Los píxeles crudos toman valores enteros entre 0 (oscuridad total) y 255 (brillo máximo). La mayor parte de los valores se concentran entre 70 y 110 (tonos de suelo y vegetación), pero aparecen picos brillantes cerca de 250 (construcciones de hormigón y techos reflectivos) y zonas cercanas a 0 (sombras profundas y cuerpos de agua).
> * **Decisión que tomamos como grupo:**
>   * Si entrenamos los modelos directamente con estos números crudos $[0, 255]$, los píxeles que tengan valores numéricos grandes (250) van a dominar con fuerza desproporcionada los cálculos matemáticos sobre los píxeles oscuros (20), sin que esto signifique necesariamente que sean más importantes. Esto provoca que el algoritmo que busca la solución (el Descenso de Gradiente) avance a los tropezones y tarde mucho más en converger.
>   * Para resolver esto, es indispensable aplicar **`StandardScaler`**: le restamos a cada píxel su promedio y lo dividimos por su dispersión, dejando todos los valores centrados en 0. De esta manera, ponemos a todos los colores a competir en igualdad de condiciones matemáticas.

## 2. Partición Estratificada de Datos y Verificación Proporcional

In [ ]:
with open(SPLITS_PATH, "r") as f:
    splits = json.load(f)

train_idx = splits["train_indices"]
dev_idx = splits["dev_indices"]
test_idx = splits["test_indices"]
targets = np.array(dataset.targets)

df_proportions = pd.DataFrame({
    "Clase": [CLASS_NAMES_ES[c] for c in CLASS_NAMES],
    "Total": [(targets == i).sum() for i in range(10)],
    "Train (80%)": [(targets[train_idx] == i).sum() for i in range(10)],
    "Dev (10%)": [(targets[dev_idx] == i).sum() for i in range(10)],
    "Test (10%)": [(targets[test_idx] == i).sum() for i in range(10)],
    "% Train": [round((targets[train_idx] == i).sum() / (targets == i).sum() * 100, 1) for i in range(10)],
    "% Dev": [round((targets[dev_idx] == i).sum() / (targets == i).sum() * 100, 1) for i in range(10)],
    "% Test": [round((targets[test_idx] == i).sum() / (targets == i).sum() * 100, 1) for i in range(10)],
})
display(df_proportions)
print("[VERIFICACIÓN EXITOSA] Cada una de las 10 clases mantiene rigurosamente el 80% en Train, 10% en Dev y 10% en Test.")

## 3. Modelo Baseline: Proceso Iterativo y Selección de Hiperparámetros
Calibramos el preprocesamiento con $N=4.000$ muestras en Train por restricción de cómputo local.

In [ ]:
MAX_TRAIN_SAMPLES = 4000
MAX_DEV_SAMPLES = 1000

def extract_flat(indices, max_samples=None):
    sub = indices[:max_samples] if max_samples is not None else indices
    X, y = [], []
    for idx in sub:
        img, target = dataset[idx]
        X.append(np.array(img, dtype=np.float32).flatten())
        y.append(target)
    return np.array(X), np.array(y)

print(f"Aplanando características (Train={MAX_TRAIN_SAMPLES}, Dev={MAX_DEV_SAMPLES})...")
X_train_flat, y_train = extract_flat(train_idx, max_samples=MAX_TRAIN_SAMPLES)
X_dev_flat, y_dev = extract_flat(dev_idx, max_samples=MAX_DEV_SAMPLES)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_dev_scaled = scaler.transform(X_dev_flat)
print(f"Dimensiones escaladas: Train={X_train_scaled.shape} | Dev={X_dev_scaled.shape}")

### 3.1 Experimento 1: Baseline por Defecto ($C = 1.0$)

In [ ]:
print("Entrenando Experimento 1: Baseline Default (C=1.0)...")
start_time_c1 = time.time()
baseline_default = LogisticRegression(max_iter=300, C=1.0, penalty='l2', solver='lbfgs', tol=1e-3, random_state=SEED)
baseline_default.fit(X_train_scaled, y_train)
time_c1 = time.time() - start_time_c1

train_acc_c1 = accuracy_score(y_train, baseline_default.predict(X_train_scaled))
dev_acc_c1 = accuracy_score(y_dev, baseline_default.predict(X_dev_scaled))

diag_exp1 = print_bias_variance_diagnosis(
    train_acc=train_acc_c1, dev_acc=dev_acc_c1, model=baseline_default, X_train=X_train_scaled,
    model_name="Experimento 1: Baseline Default (C=1.0)"
)

### 3.2 Análisis Crítico del Régimen $P \gg N$ (La Trampa de la Memorización)
Con $C=1.0$, los $122.880$ pesos libres memorizan las $4.000$ imágenes de Train ($99.5\%$ acierto) pero colapsan en Dev ($32\%$). Imponemos $C = 0.01$ para forzar regularización $L_2$ estricta.

### 3.3 Experimento 2: Baseline Calibrado con Regularización $L_2$ ($C = 0.01$)

In [ ]:
print("Entrenando Experimento 2: Baseline Regularizado (C=0.01)...")
start_time_c01 = time.time()
baseline_regularized = LogisticRegression(max_iter=300, C=0.01, penalty='l2', solver='lbfgs', tol=1e-3, random_state=SEED)
baseline_regularized.fit(X_train_scaled, y_train)
time_c01 = time.time() - start_time_c01

y_train_pred = baseline_regularized.predict(X_train_scaled)
y_dev_pred = baseline_regularized.predict(X_dev_scaled)

train_acc = accuracy_score(y_train, y_train_pred)
dev_acc = accuracy_score(y_dev, y_dev_pred)
macro_f1 = f1_score(y_dev, y_dev_pred, average='macro')
latency_ms = (time_c01 / len(X_dev_scaled)) * 1000.0

diag_exp2 = print_bias_variance_diagnosis(
    train_acc=train_acc, dev_acc=dev_acc, model=baseline_regularized, X_train=X_train_scaled,
    model_name="Experimento 2: Baseline Calibrado (C=0.01)"
)

## 4. Evaluación Rigurosa del Baseline Definitivo

In [ ]:
print("=" * 60)
print(f"Exactitud en Train: {train_acc * 100:.2f}% | Exactitud en Dev: {dev_acc * 100:.2f}%")
print(f"Métrica de Optimización (Macro F1): {macro_f1:.4f} ({macro_f1*100:.2f}%)")
print(f"Latencia de Inferencia Estimada:    {latency_ms:.2f} ms")
print("=" * 60)

In [ ]:
# Matrices de Confusión Separadas
cm_abs = confusion_matrix(y_dev, y_dev_pred)
plt.figure(figsize=(9, 7.5))
sns.heatmap(cm_abs, annot=True, fmt='d', cmap='Blues', square=True,
            cbar_kws={'shrink': 0.75, 'label': 'Muestras'},
            xticklabels=[CLASS_NAMES_ES[c] for c in CLASS_NAMES],
            yticklabels=[CLASS_NAMES_ES[c] for c in CLASS_NAMES], annot_kws={"size": 9})
plt.title("Matriz de Confusión: Conteos Absolutos (Dev Set)", fontsize=13, pad=12, fontweight='bold')
plt.xlabel("Clase Predicha")
plt.ylabel("Clase Real")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

cm_norm = cm_abs.astype('float') / cm_abs.sum(axis=1)[:, np.newaxis]
plt.figure(figsize=(9, 7.5))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', square=True, vmin=0.0, vmax=1.0,
            cbar_kws={'shrink': 0.75, 'label': 'Tasa de Acierto (Recall)'},
            xticklabels=[CLASS_NAMES_ES[c] for c in CLASS_NAMES],
            yticklabels=[CLASS_NAMES_ES[c] for c in CLASS_NAMES], annot_kws={"size": 9})
plt.title("Matriz de Confusión Normalizada: Recall por Fila (Dev Set)", fontsize=13, pad=12, fontweight='bold')
plt.xlabel("Clase Predicha")
plt.ylabel("Clase Real")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Comparación Experimental y Diagnóstico Consolidado

In [ ]:
df_comparison = pd.DataFrame([
    {
        "Experimento": "1. Baseline Default (C=1.0)",
        "Fuerza L2": "Débil (λ=1)",
        "Train Acc": f"{train_acc_c1 * 100:.2f}%",
        "Dev Acc": f"{dev_acc_c1 * 100:.2f}%",
        "Brecha (Gap)": f"{(train_acc_c1 - dev_acc_c1) * 100:.2f}%",
        "Diagnóstico": diag_exp1["diagnosis"]
    },
    {
        "Experimento": "2. Baseline Calibrado (C=0.01)",
        "Fuerza L2": "Estricta (λ=100)",
        "Train Acc": f"{train_acc * 100:.2f}%",
        "Dev Acc": f"{dev_acc * 100:.2f}%",
        "Brecha (Gap)": f"{(train_acc - dev_acc) * 100:.2f}%",
        "Diagnóstico": diag_exp2["diagnosis"]
    }
])
display(df_comparison)

## 6. Conclusiones Oficiales de la Etapa 1 y Hoja de Ruta hacia la Etapa 2
1. **Ciclo Iterativo Validado:** Se comprobó que el Baseline sin regularizar ($C=1.0$) memoriza el conjunto de entrenamiento por sobreparametrización ($P=122.890 \gg N=4.000$).
2. **Sesgo Alto Estructural (Underfitting):** Al calibrar la regularización $L_2$ ($C=0.01$), se eliminó la memorización espuria y emergió el verdadero límite del modelo lineal ($\text{Train} \approx 41\%$, $\text{Dev} \approx 36\%$, $\text{Gap} \approx 5\%$).
3. **Punto de Referencia Inmutable:** La métrica oficial de referencia es **Macro F1 = 0.3524** en Dev.
4. **Hoja de Ruta hacia la Etapa 2:**  
   * Análisis clínico de 50 imágenes mal clasificadas en Dev.
   * Implementación de un **Perceptrón Multicapa (MLP)** de 3 capas ocultas con activaciones **GELU/ReLU** para quebrar la barrera del sesgo lineal.